In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaConfig
import math
import collections
import gc
import numpy as np


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cleanup_memory():
    gc.collect()
    torch.cuda.empty_cache()


class LSH:
    """
    Standard LSH implementation using fixed K-bit hash buckets.
    Equivalent to L independent hash tables with K bits each.
    """
    def __init__(self, K, L, num_heads, num_kv_heads, device='cpu'):
        self.tables = []
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.device = device
        self.K = K
        self.L = L
        self.tables = [[collections.defaultdict(list) for _ in range(L)] for _ in range(num_kv_heads)]

    def clear(self):
        for head_tables in self.tables:
            for table in head_tables:
                table.clear()

    def fill(self, hash_codes, indices):
        num_kv, N, L, K = hash_codes.shape
        idx = indices.cpu().tolist()

        powers = 2 ** torch.arange(K, device=self.device).float()

        buckets = (hash_codes * powers).sum(dim=-1).long().cpu()
        
        for h in range(num_kv):
            for l in range(L):
                table_dict = self.tables[h][l]
                row_buckets = buckets[h, :, l].tolist()
                for i, val in enumerate(row_buckets):
                    table_dict[val].append(idx[i])

    def batch_retrieve(self, query_hash_codes):

        num_heads, L, K = query_hash_codes.shape
        num_groups = num_heads // self.num_kv_heads
        
        powers = 2 ** torch.arange(K, device=self.device).float()
        q_buckets = (query_hash_codes * powers).sum(dim=-1).long().cpu() # [H, L]
        
        results = []
        for h in range(num_heads):
            kv_head = h // num_groups
            counts = collections.defaultdict(int)
            
            for l in range(L):
                val = q_buckets[h, l].item()
                bucket = self.tables[kv_head][l].get(val, [])
                for idx in bucket:
                    counts[idx] += 1
            
            # switch to using 1 for comparison puprose
            candidates = [idx for idx, count in counts.items() if count >= 1]
            
            if candidates:
                results.append(torch.tensor(candidates, dtype=torch.long, device=self.device))
            else:
                results.append(torch.empty(0, dtype=torch.long, device=self.device))
        return results

# ==========================================
# Part 3: Sparse Attention Kernels (Math)
# ==========================================

def _compute_core_attention(q, k, v, k_norm, head_dim):
    """
    Computes standard Dot-Product Attention on the selected subset.
    Returns score_f (raw logits) and theta (angular distance) for SNIS.
    """
    if k.shape[0] == 0:
        return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype), None, None

    # Score = Q * K^T
    score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
    
    # Compute angle theta for SNIS weight calculation
    q_norm = q.float().norm(p=2)
    denom = q_norm * k_norm.float()
    cos_theta = torch.clamp(score_f / (denom + 1e-6), -1.0 + 1e-4, 1.0 - 1e-4)
    theta = torch.acos(cos_theta)
    
    return score_f, theta

def _apply_snis_and_project(score_f, log_w, v, head_dim):
    """
    Applies Self-Normalized Importance Sampling (SNIS).
    Formula: Output = sum( (exp(score)/w) * v ) / sum( exp(score)/w )
    Log Space: log_coeff = score - log_w
    """
    scaled_logits = score_f / math.sqrt(head_dim) - log_w
    attn_probs = torch.softmax(scaled_logits, dim=0)
    
    return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

def magicpig_transform(q, k, v, k_norm, K, L, head_dim):
    """
    Standard MagicPIG Estimator.
    Assumes fixed sampling probability based on K-bit collision.
    """
    score_f, theta = _compute_core_attention(q, k, v, k_norm, head_dim)
    if theta is None: return score_f

    # 1. Probability of collision for a single hash function
    prob_bit = 1.0 - theta / math.pi
    
    # 2. Probability of collision for K bits (Bucket Match)
    p_bucket = prob_bit.pow(K) 
    
    # 3. Probability of retrieval (collision in at least 1 of L tables)
    u = 1.0 - (1.0 - p_bucket).pow(L)
    log_w = torch.log(torch.clamp(u, min=1e-8))

    return _apply_snis_and_project(score_f, log_w, v, head_dim)

def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim):
    """
    Jungle Estimator.
    Uses ACTUAL retrieval depth (delta) per token to compute importance weight.
    Stabilizes variance when backtracking occurs (delta < K).
    """
    score_f, theta = _compute_core_attention(q, k, v, k_norm, head_dim)
    if theta is None: return score_f

    prob_bit = 1.0 - theta / math.pi
    
    # [JUNGLE LOGIC]
    # If we backtracked to K-1, retrieval_depths is K-1.
    # This results in higher p_collision, thus lower weight `u`, correcting the bias.
    p_collision = prob_bit.pow(retrieval_depths.float())
    
    u = 1.0 - (1.0 - p_collision).pow(L)
    log_w = torch.log(torch.clamp(u, min=1e-8))

    return _apply_snis_and_project(score_f, log_w, v, head_dim)

# ==========================================
# Part 4: The Server Wrapper
# ==========================================

class LSHSparseAttnServer:
    def __init__(self, config, K=8, L=20, batch_size=1, max_length=2048, 
                 device='cpu', dtype=torch.float32, use_jungle=False):
        self.K, self.L = K, L
        self.batch_size = batch_size
        self.device, self.dtype = device, dtype
        self.use_jungle = use_jungle
        
        self.jg_K_max = K
        self.jg_L = L

        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.hidden_size // self.num_heads
        
        # KV Caches [B, H, S, D]
        self.k_cache = torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype)
        self.v_cache = torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype)
        self.avg_k_cache = torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype)
        self.current_len = [0] * batch_size

        # LSH Components
        self.lsh = LSH(K, L, self.num_heads, self.num_kv_heads, device)
        # Random projection matrix for hashing
        self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
        
        # Jungle Cache: Stores raw bits [N, L, K] for tree traversal
        self.jg_hash_cache = {} 

    def fill(self, request_id, key_states, value_states):
        """Processes keys/values, computes hashes, and fills tables/caches."""
        seq_len = key_states.shape[0]
        start_pos = self.current_len[request_id]
        end_pos = start_pos + seq_len
        
        # 1. Update Dense Cache
        self.k_cache[request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
        self.v_cache[request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
        self.current_len[request_id] = end_pos

        # 2. Compute Hashes
        keys = self.k_cache[request_id, :, start_pos:end_pos, :]
        avg_k = keys.float().mean(dim=1, keepdim=True).to(self.dtype)
        self.avg_k_cache[request_id] = avg_k
        centered = keys - avg_k

        # Projection: [H, N, D] @ [D, LK] -> [H, N, LK]
        projections = torch.matmul(centered, self.hash_func)
        bits = (projections > 0).float() # [H, N, LK]

        if self.use_jungle:
            # Store raw bits for tree traversal
            bits_reshaped = bits.view(self.num_kv_heads, -1, self.jg_L, self.jg_K_max)
            if request_id not in self.jg_hash_cache:
                self.jg_hash_cache[request_id] = bits_reshaped
            else:
                self.jg_hash_cache[request_id] = torch.cat([self.jg_hash_cache[request_id], bits_reshaped], dim=1)
        else:
            # Fill fixed LSH tables
            bits_reshaped = bits.view(self.num_kv_heads, -1, self.L, self.K)
            self.lsh.fill(bits_reshaped, torch.arange(start_pos, end_pos, device=self.device))

    def decode(self, query_states):
        """Retrieves keys and computes attention using Sparse estimators."""
        bsz, n_heads, _, dim = query_states.shape
        req_id = 0 # Simplified for toy problem
        q_heads = query_states[req_id, :, 0, :] # [n_heads, dim]
        
        # Project Query
        norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)
        q_projections = torch.matmul(norm_q, self.hash_func)
        q_bits_raw = (q_projections > 0).float()

        if self.use_jungle:
            jg_q_bits = q_bits_raw.view(self.num_heads, self.jg_L, self.jg_K_max)
        else:
            bits = q_bits_raw.view(self.num_heads, self.L, self.K)
            idx_list = self.lsh.batch_retrieve(bits)

        head_outputs = []
        num_groups = n_heads // self.num_kv_heads

        for h in range(n_heads):
            kv_head = h // num_groups
            sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
            retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

            # [Jungle Strategy]
            if self.use_jungle:
                k_bits = self.jg_hash_cache[req_id][kv_head] # [N, L, K]
                q_b = jg_q_bits[h] # [L, K]

                # 1. Calculate Prefix Match Length (Depth)
                match_matrix = (k_bits == q_b.unsqueeze(0)).int()
                # Cumprod: 1s until first 0. Sum gives depth.
                depths = match_matrix.cumprod(dim=-1).sum(dim=-1) # [N, L]

                # 2. Backtracking Logic
                full_match_mask = (depths == self.jg_K_max) # [N, L]
                
                # Does this tree have *any* confident matches at max depth?
                tree_found_exact = full_match_mask.any(dim=0) # [L]
                
                # If YES: Use mask at K. If NO: Use mask at K-1.
                backtrack_mask = (depths >= (self.jg_K_max - 1))
                final_mask = torch.where(tree_found_exact.unsqueeze(0), full_match_mask, backtrack_mask)

                # Select unique keys
                is_selected = final_mask.any(dim=1)
                sparse_indices = torch.nonzero(is_selected).squeeze(-1)

                if sparse_indices.numel() > 0:
                    # Assign SNIS depths for correct weighting
                    selected_depths = depths[sparse_indices]
                    selected_mask = final_mask[sparse_indices]
                    # Filter for the specific depth used for selection
                    valid_depths = selected_depths * selected_mask.int()
                    retrieved_depths = valid_depths.max(dim=1).values.float()

            # [MagicPIG Strategy]
            else:
                sparse_indices = idx_list[h]

            # [Compute Attention]
            if sparse_indices.numel() == 0:
                head_outputs.append(torch.zeros(1, self.head_dim, device=self.device, dtype=self.dtype))
                continue

            # Gather Data
            k_sel = self.k_cache[req_id, kv_head, sparse_indices, :]
            v_sel = self.v_cache[req_id, kv_head, sparse_indices, :]
            
            k_sel_centered = k_sel - self.avg_k_cache[req_id, kv_head, 0, :]
            k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)

            if self.use_jungle:
                out_h = jungle_snis_transform(
                    q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, 
                    retrieved_depths, self.jg_L, self.head_dim
                )
            else:
                out_h = magicpig_transform(
                    q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, 
                    self.K, self.L, self.head_dim
                )
            
            head_outputs.append(out_h)

        return torch.cat(head_outputs, dim=1)

# ==========================================
# Part 5: The Experiment
# ==========================================

def run_toy_problem():
    print("="*60)
    print("JUNGLE vs MAGICPIG: Sparse Attention Toy Problem")
    print("="*60)

    # 1. Setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    config = LlamaConfig(hidden_size=64, num_attention_heads=4, num_key_value_heads=4)
    
    # Parameters designed to create a failure case for Fixed LSH (MagicPIG)
    # High K means high precision but low recall.
    K = 8       
    L = 25      
    print(f"Config: K={K}, L={L}, Device={device}")

    # 2. Create Data: "Needle in a Haystack"
    # Angle theta chosen such that Prob(match K bits) is low (<5%), 
    # but Prob(match K-1) is decent (~10%).
    target_sim = 0.65
    theta = (1 - target_sim) * math.pi
    
    head_dim = 16
    
    # Base Query Vector
    q_vec = torch.randn(1, head_dim, device=device)
    q_vec = q_vec / q_vec.norm()
    
    # Needle Key (Rotated Query) - Hard to find!
    rand_ortho = torch.randn(1, head_dim, device=device)
    rand_ortho = rand_ortho - torch.dot(rand_ortho.view(-1), q_vec.view(-1)) * q_vec
    rand_ortho = rand_ortho / rand_ortho.norm()
    
    k_needle = math.cos(theta) * q_vec + math.sin(theta) * rand_ortho
    v_needle = torch.ones_like(k_needle) * 100.0 # Distinctive value to track
    
    # Distractor Keys (Noise)
    n_distractors = 200
    k_noise = torch.randn(n_distractors, head_dim, device=device)
    k_noise = k_noise / k_noise.norm(dim=-1, keepdim=True)
    v_noise = torch.randn(n_distractors, head_dim, device=device)
    
    # Assemble Batch
    keys = torch.cat([k_noise, k_needle], dim=0).unsqueeze(1).repeat(1, 4, 1)
    vals = torch.cat([v_noise, v_needle], dim=0).unsqueeze(1).repeat(1, 4, 1)
    
    print(f"Needle inserted at index {n_distractors} with similarity {math.cos(theta):.3f}")
    print(f"Expected P(Exact Match K={K}): {(1-theta/math.pi)**K:.4f}")
    print(f"Expected P(Relaxed Match K={K-1}): {(1-theta/math.pi)**(K-1):.4f}")
    
    # 3. Run MagicPIG (Standard)
    print("\nRunning MagicPIG (Fixed Depth)...")
    server_pig = LSHSparseAttnServer(config, K=K, L=L, device=device, use_jungle=False)
    server_pig.fill(0, keys, vals)
    
    q_input = q_vec.view(1, 1, 1, head_dim).repeat(1, 4, 1, 1)
    out_pig = server_pig.decode(q_input)
    
    # 4. Run Jungle (Forest)
    print("Running Jungle (Adaptive Backtracking)...")
    server_jg = LSHSparseAttnServer(config, K=K, L=L, device=device, use_jungle=True)
    server_jg.hash_func = server_pig.hash_func # Force same hashes for fair comparison
    server_jg.fill(0, keys, vals)
    
    out_jg = server_jg.decode(q_input)
    
    # 5. Analysis
    # We check the output magnitude. If the needle (value=100) was found, magnitude is large.
    weight_pig = out_pig[0, :head_dim].mean().item() 
    weight_jg = out_jg[0, :head_dim].mean().item()
    
    print("\n" + "-"*30)
    print(f"Recovered Signal Strength (Target ~10.0):")
    print(f"  MagicPIG: {weight_pig:.4f}")
    print(f"  Jungle:   {weight_jg:.4f}")
    print("-" * 30)
    
    if weight_jg > weight_pig * 1.5:
        print("✅ SUCCESS: Jungle retrieved the needle! MagicPIG missed it.")
        print("   The 'step back' logic successfully caught the noisy key.")
    elif weight_jg > 0.1 and weight_pig > 0.1:
        print("⚠️  Both methods retrieved it. Try increasing noise (lower target_sim).")
    else:
        print("❌ Both missed. Try decreasing K or increasing L.")
    
    cleanup_memory()

if __name__ == "__main__":
    set_seed(42)
    run_toy_problem()

JUNGLE vs MAGICPIG: Sparse Attention Toy Problem
Config: K=8, L=25, Device=cpu
Needle inserted at index 200 with similarity 0.454
Expected P(Exact Match K=8): 0.0319
Expected P(Relaxed Match K=7): 0.0490

Running MagicPIG (Fixed Depth)...
Running Jungle (Adaptive Backtracking)...

------------------------------
Recovered Signal Strength (Target ~10.0):
  MagicPIG: 0.0003
  Jungle:   1.3367
------------------------------
✅ SUCCESS: Jungle retrieved the needle! MagicPIG missed it.
   The 'step back' logic successfully caught the noisy key.
